# FinReasoning — standalone evaluation

Run this notebook to evaluate a fine-tuned adapter **with valid metrics**.

**Why Step 4 in the main Colab notebook looked broken:** `data/processed` stores only `prompt`, `completion`, and `task` for SFT. The evaluator needs `answer`, `question`, `context`, and (for numerical tasks) `expression` / `variables`. Loading the test split from disk dropped those fields, so prompts were empty, predictions became *Insufficient information.*, and ground truth appeared empty (NaN in CSV).

**This notebook** rebuilds the **test split from raw JSONL** (same stratified split as preprocessing) and uses the fixed evaluator that builds prompts with `format_as_prompt_completion` (matching training).

**Requirements:** GPU recommended; raw training JSONL under `data/raw/` (or set `RAW_DATA_PATH`); adapter at `outputs/sft_qlora/final_adapter` or set `ADAPTER_DIR`.

**Google Colab:** If you use Drive, run the optional **Google Colab** cell below first. It clones [github.com/juankim834/FinReasoningAI](https://github.com/juankim834/FinReasoningAI) into the same Drive workspace as this notebook (same layout as `FinReasoningAI_Colab.ipynb` Step 0).

## Google Colab (optional)

Mount Drive and clone or update **[juankim834/FinReasoningAI](https://github.com/juankim834/FinReasoningAI)** on **Google Drive in the same folder as this notebook** (alongside `FinReasoningAI_Eval.ipynb`). Skip this section if you already run from a local git checkout.

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    print("Not on Google Colab — skipping Drive clone (use a local repo checkout).")
else:
    from pathlib import Path
    import os
    import sys

    from google.colab import drive

    drive.mount("/content/drive")

    REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
    DRIVE_FALLBACK = "/content/drive/MyDrive/FinReasoningAI"
    NB_NAMES = ("FinReasoningAI_Eval.ipynb", "FinReasoningAI_Colab.ipynb")

    def _infer_notebook_workspace():
        root = Path("/content/drive/MyDrive")
        if not root.is_dir():
            return None
        candidates = []
        for nb in NB_NAMES:
            if (root / nb).is_file():
                candidates.append(root.resolve())
        for child in sorted(root.iterdir()):
            if not child.is_dir():
                continue
            for nb in NB_NAMES:
                if (child / nb).is_file():
                    candidates.append(child.resolve())
                    break
            nested = child / "FinReasoningAI"
            if nested.is_dir():
                for nb in NB_NAMES:
                    if (nested / nb).is_file():
                        candidates.append(nested.resolve())
                        break
        seen = set()
        uniq = []
        for c in candidates:
            s = str(c)
            if s not in seen:
                seen.add(s)
                uniq.append(c)
        if len(uniq) == 1:
            return str(uniq[0])
        if len(uniq) > 1:
            print(
                "[WARN] Multiple notebook paths on Drive; using DRIVE_FALLBACK. "
                "Set WORKSPACE manually in this cell."
            )
        return None

    WORKSPACE = _infer_notebook_workspace() or DRIVE_FALLBACK
    os.makedirs(WORKSPACE, exist_ok=True)

    if os.path.isdir(os.path.join(WORKSPACE, ".git")):
        PROJECT_DIR = WORKSPACE
    else:
        PROJECT_DIR = os.path.join(WORKSPACE, "FinReasoningAI")

    import subprocess

    if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
        print(f"Cloning {REPO_URL} -> {PROJECT_DIR}")
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    else:
        print(f"Repo at {PROJECT_DIR}; pulling latest...")
        subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    print("Working directory:", os.getcwd())

In [ ]:
import os, sys, gc
from pathlib import Path

# Repo root (directory that contains `src/`)
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / "src").is_dir():
            ROOT = p.resolve()
            break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

## Configuration

Edit paths if needed. On Colab, upload your `data/raw/*.jsonl` and adapter folder, or mount Drive and set paths accordingly.

In [ ]:
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-14B-Instruct")
# Directory of *.jsonl or a single .jsonl file — same source used for `load_and_format_dataset` / preprocessing
RAW_DATA_PATH = os.environ.get("RAW_DATA_PATH", "data/raw")
ADAPTER_DIR = os.environ.get("ADAPTER_DIR", "outputs/sft_qlora/final_adapter")

OUTPUT_CSV = "outputs/eval_results.csv"
MAX_SAMPLES = None  # e.g. 50 for a quick smoke test; None = all test samples
MAX_NEW_TOKENS = 128

_raw = Path(RAW_DATA_PATH)
assert _raw.exists(), (
    f"Raw data not found at {RAW_DATA_PATH}. "
    "Place your JSONL files there or set RAW_DATA_PATH to your combined training file."
)
assert Path(ADAPTER_DIR).exists(), (
    f"Adapter not found at {ADAPTER_DIR}. Train SFT first or set ADAPTER_DIR."
)

print("MODEL_ID:", MODEL_ID)
print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("ADAPTER_DIR:", ADAPTER_DIR)

In [ ]:
from src.data.preprocess import load_eval_test_samples

test_samples = load_eval_test_samples(RAW_DATA_PATH)
print(f"Test split: {len(test_samples)} samples")
if test_samples:
    s0 = test_samples[0]
    print("Example keys:", sorted(s0.keys()))
    print("task:", s0.get("task"))
    print("answer (prefix):", str(s0.get("answer", ""))[:120])

In [ ]:
import torch
from peft import PeftModel
from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    import flash_attn  # noqa: F401
    _attn = "flash_attention_2"
except ImportError:
    _attn = "eager"

eval_base, eval_tokenizer = load_model_and_tokenizer(
    MODEL_ID, DEFAULT_BNB_CONFIG, attn_implementation=_attn
)
eval_model = PeftModel.from_pretrained(eval_base, ADAPTER_DIR)
eval_model.eval()

if torch.cuda.is_available():
    print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("Model loaded (CPU — generation will be slow).")

In [ ]:
from src.eval.evaluate import evaluate_model

metrics = evaluate_model(
    model=eval_model,
    tokenizer=eval_tokenizer,
    test_dataset=test_samples,
    output_csv=OUTPUT_CSV,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=MAX_SAMPLES,
)

print("\nEvaluation Results:")
print("-" * 40)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")

In [ ]:
import pandas as pd

results_df = pd.read_csv(OUTPUT_CSV)
# Avoid pandas inferring empty columns as numeric NaN
for col in ("question", "ground_truth", "prediction"):
    if col in results_df.columns:
        results_df[col] = results_df[col].fillna("").astype(str)

print(f"Total evaluated: {len(results_df)} samples\n")
print("Per-task breakdown:")
print(results_df.groupby("task")[["exact_match", "f1", "parsable", "grounding_rate"]].mean().round(4))

bad = results_df[results_df["exact_match"] == 0]
print("\nLow-scoring samples (EM=0), first 15:")
display(bad[["task", "question", "ground_truth", "prediction", "f1", "grounding_rate"]].head(15))